# Transformation gate ablation

## Goal

Diagnose where PRISM transformation candidates are lost, one condition at a time, without changing production code or defaults.

This notebook keeps every initial candidate in a ledger and provides four complementary views:

1. independent gate outcomes over the full initial cohort;
2. cumulative attrition in a scientifically interpretable diagnostic order;
3. a replay of the current production order in `src/transformation.py`;
4. leave-one-gate-out rescue counts.

**Important:** this is an ablation/observability notebook, not evidence that a production threshold should be relaxed. The retained example contains only one query pair and one template.

## Setup

The default inputs are a frozen retained TMalign run. Protocol assets are read explicitly from the current canonical template root because the isolated bundle's local protocol-asset directories are empty. No retained input is modified.

### Key assumptions

- Template ID `1kcaCH` means chain C paired with chain H.
- Alignment files without an `aligner` field belong to the explicitly selected TMalign run.
- Interface coverage follows the live strict comparison: `>50%` for template interfaces of at most 50 residues, otherwise `>30%` with current defaults.
- Hotspot and complementary-contact gates are active only in `published_protocol`; their measurements remain visible in either mode.
- Transformation materialization uses the retained surface-extraction PDBs and writes only to a temporary directory.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

SOURCE_REPO = Path("/scratch/rshadi25/GitHub/PRISM-prescript")
RUN_ROOT = Path("/scratch/tmp/prism-prescript-pipeline-extension-clean/runs/20260816T121957Z-tmalign")
ASSET_ROOT = SOURCE_REPO / "new_template/template"
TEMPLATE_ID = "	1h7xCD"
TEMPLATE_CHAINS = ("C", "D")
RUN_ALIGNER = "tmalign"

# Change one parameter at a time, then restart the kernel and Run All.
FILTER_MODE = "published_protocol"  # or "geometry_only_experimental"

assert FILTER_MODE in {"published_protocol", "geometry_only_experimental"}
if str(SOURCE_REPO) not in sys.path:
    sys.path.insert(0, str(SOURCE_REPO))

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

### 1. Validate and fingerprint inputs

In [2]:
from src import transformation
from src.template_filtering import (
    count_matched_complementary_contacts,
    evaluate_hotspots,
    load_filter_assets,
)
from src.utils import distance_calculator, read_ca_coordinates

alignment_dir = RUN_ROOT / "processed/alignment"
required_paths = [
    SOURCE_REPO / "src/transformation.py",
    SOURCE_REPO / "src/template_filtering.py",
    RUN_ROOT / "inputs.csv",
    ASSET_ROOT / "interfaces_lists" / f"{TEMPLATE_ID}.json",
    ASSET_ROOT / "hotspots" / f"{TEMPLATE_ID}.json",
    ASSET_ROOT / "contacts" / f"{TEMPLATE_ID}.json",
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing required inputs: {missing_paths}"

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

provenance = pd.DataFrame(
    {"path": str(path), "bytes": path.stat().st_size, "sha256": sha256(path)}
    for path in required_paths
)
display(provenance)

,path,bytes,sha256
0,/scratch/rshadi25/GitHub/PRISM-prescript/src/t...,20138,520dcad61a69f05e0c52e6541c2968a16d95d11d683741...
1,/scratch/rshadi25/GitHub/PRISM-prescript/src/t...,7347,8050ca92049d9cdc1b85082b3850a4f6ce732f4d812ae5...
2,/scratch/tmp/prism-prescript-pipeline-extensio...,28,d1409ae4f385421080d2be8a2a6612693d1540f5820129...
3,/scratch/rshadi25/GitHub/PRISM-prescript/new_t...,725,0b5a045ba497c2259efa8d37cbd3159062939247750bbe...
4,/scratch/rshadi25/GitHub/PRISM-prescript/new_t...,2762,c241c41042498ac8567311dbed3e3e7a8b83fe02bfd38a...
5,/scratch/rshadi25/GitHub/PRISM-prescript/new_t...,204,2ea8dac2930cf79bc204c3bccc39a2275ee2c6348127e2...


### 2. Freeze the live thresholds

In [3]:
thresholds = {
    "minimum_match_count": transformation.MINIMUM_RESIDUE_MATCH_COUNT,
    "minimum_match_percentage": transformation.MINIMUM_RESIDUE_MATCH_PERCENTAGE,
    "difference_allowance_percentage": transformation.DIFF_PERCENTAGE,
    "template_residue_boundary": transformation.TEMPLATE_RESIDUE_COUNT,
    "tm_score_threshold": transformation.TM_SCORE_THRESHOLD,
    "minimum_hotspots_per_side": transformation.MINIMUM_HOTSPOT_MATCH_NUMBER,
    "minimum_complementary_contacts": transformation.CONTACT_COUNT_THRESHOLD,
    "clashing_distance_angstrom": transformation.CLASHING_DISTANCE,
    "maximum_clash_count": transformation.MAX_CLASHING_COUNT,
    "source_default_filter_mode": transformation.FILTER_MODE,
    "notebook_filter_mode": FILTER_MODE,
}
display(pd.Series(thresholds, name="value").to_frame())

,value
minimum_match_count,15
minimum_match_percentage,50.0
difference_allowance_percentage,20.0
template_residue_boundary,50
tm_score_threshold,0.5
minimum_hotspots_per_side,1
minimum_complementary_contacts,5
clashing_distance_angstrom,3.0
maximum_clash_count,5
source_default_filter_mode,geometry_only_experimental


## Steps

### 3. Load candidates and protocol assets

The two possible orientations are retained as separate candidates:

- `o1`: receptor → C and ligand → H
- `o2`: receptor → H and ligand → C

In [4]:
inputs = pd.read_csv(RUN_ROOT / "inputs.csv")
assert len(inputs) == 1, "This demonstration expects one frozen query pair."
left_query = str(inputs.iloc[0]["Receptor"])
right_query = str(inputs.iloc[0]["Ligand"])

interface_lists = json.loads(
    (ASSET_ROOT / "interfaces_lists" / f"{TEMPLATE_ID}.json").read_text()
)
filter_assets = load_filter_assets(TEMPLATE_ID, ASSET_ROOT)

def load_alignment(query: str, chain: str) -> dict:
    path = alignment_dir / f"{query}_{TEMPLATE_ID}_{chain}.json"
    if not path.is_file():
        return {"_path": str(path), "_available": False}
    record = json.loads(path.read_text())
    record["_path"] = str(path)
    record["_available"] = True
    record["_effective_aligner"] = str(record.get("aligner") or RUN_ALIGNER).lower()
    return record

candidates = [
    {
        "candidate_id": f"{TEMPLATE_ID}:{left_query}:{right_query}:o1",
        "orientation": "o1",
        "left_query": left_query,
        "left_chain": TEMPLATE_CHAINS[0],
        "right_query": right_query,
        "right_chain": TEMPLATE_CHAINS[1],
    },
    {
        "candidate_id": f"{TEMPLATE_ID}:{left_query}:{right_query}:o2",
        "orientation": "o2",
        "left_query": left_query,
        "left_chain": TEMPLATE_CHAINS[1],
        "right_query": right_query,
        "right_chain": TEMPLATE_CHAINS[0],
    },
]
for candidate in candidates:
    candidate["left_alignment"] = load_alignment(candidate["left_query"], candidate["left_chain"])
    candidate["right_alignment"] = load_alignment(candidate["right_query"], candidate["right_chain"])

candidate_inputs = pd.DataFrame(
    {
        "candidate_id": c["candidate_id"],
        "orientation": c["orientation"],
        "left_chain": c["left_chain"],
        "right_chain": c["right_chain"],
        "left_match_count": c["left_alignment"].get("match_count"),
        "right_match_count": c["right_alignment"].get("match_count"),
        "left_tm_score": c["left_alignment"].get("tm_score"),
        "right_tm_score": c["right_alignment"].get("tm_score"),
    }
    for c in candidates
)
display(candidate_inputs)
print("Protocol asset format:", filter_assets["asset_format"])
print("Hotspot SHA256:", filter_assets["hotspots_sha256"])
print("Contact SHA256:", filter_assets["contacts_sha256"])

,candidate_id,orientation,left_chain,right_chain,left_match_count,right_match_count,left_tm_score,right_tm_score
0,1kcaCH:1FGNH:1TFHA:o1,o1,C,H,17,19,0.24791,0.30426
1,1kcaCH:1FGNH:1TFHA:o2,o2,H,C,18,25,0.27971,0.27712


Protocol asset format: modern_json
Hotspot SHA256: c241c41042498ac8567311dbed3e3e7a8b83fe02bfd38a86bc12806969270f3d
Contact SHA256: 2ea8dac2930cf79bc204c3bccc39a2275ee2c6348127e27a2682167d3c229679


### 4. Define observable gate measurements

In [5]:
PASS = "pass"
FAIL = "fail"
NOT_EVALUABLE = "not_evaluable"
NOT_APPLIED = "not_applied"

def transform_fields_valid(alignment: dict) -> bool:
    translation = alignment.get("translation")
    rotation = alignment.get("rotation_mat")
    return (
        isinstance(translation, list)
        and len(translation) == 3
        and isinstance(rotation, list)
        and len(rotation) == 3
        and all(isinstance(row, list) and len(row) == 3 for row in rotation)
    )

def coverage_requirement(template_size: int) -> float:
    if template_size > transformation.TEMPLATE_RESIDUE_COUNT:
        return transformation.MINIMUM_RESIDUE_MATCH_PERCENTAGE - transformation.DIFF_PERCENTAGE
    return transformation.MINIMUM_RESIDUE_MATCH_PERCENTAGE

def score_passes(alignment: dict) -> bool:
    # Call the live aligner-specific contract. Missing aligner means TMalign here.
    record = dict(alignment)
    record["aligner"] = record.get("aligner") or RUN_ALIGNER
    return transformation.alignment_score_passes(record)

def count_ca_clashes(left_path: Path, right_path: Path) -> int:
    left_coords = read_ca_coordinates(str(left_path))
    right_coords = read_ca_coordinates(str(right_path))
    return sum(
        distance_calculator(left_coord, right_coord) < transformation.CLASHING_DISTANCE
        for left_coord in left_coords
        for right_coord in right_coords
    )

def materialize_candidate(candidate: dict) -> dict:
    left_input = RUN_ROOT / "processed/surface_extraction" / f"{candidate['left_query']}.asa.pdb"
    right_input = RUN_ROOT / "processed/surface_extraction" / f"{candidate['right_query']}.asa.pdb"
    if not left_input.is_file() or not right_input.is_file():
        return {"status": NOT_EVALUABLE, "reason": "surface_input_missing", "clash_count": None}
    with tempfile.TemporaryDirectory(prefix="prism-transform-ablation-") as tmp:
        left_output = Path(tmp) / "left.pdb"
        right_output = Path(tmp) / "right.pdb"
        left_ok = transformation.apply_tm_transform(
            str(left_input), str(left_output),
            candidate["left_alignment"].get("translation", []),
            candidate["left_alignment"].get("rotation_mat", []),
        )
        right_ok = transformation.apply_tm_transform(
            str(right_input), str(right_output),
            candidate["right_alignment"].get("translation", []),
            candidate["right_alignment"].get("rotation_mat", []),
        )
        if not left_ok or not right_ok or not left_output.is_file() or not right_output.is_file():
            return {"status": FAIL, "reason": "transform_write_failed", "clash_count": None}
        return {
            "status": PASS,
            "reason": "temporary_outputs_written",
            "clash_count": count_ca_clashes(left_output, right_output),
        }

### 5. Evaluate every gate independently

In [6]:
gate_rows = []
candidate_metrics = {}

def add_gate(candidate_id, gate, status, observed, criterion, active=True):
    gate_rows.append(
        {
            "candidate_id": candidate_id,
            "gate": gate,
            "status": status,
            "observed": observed,
            "criterion": criterion,
            "active_in_selected_mode": active,
        }
    )

for candidate in candidates:
    cid = candidate["candidate_id"]
    left = candidate["left_alignment"]
    right = candidate["right_alignment"]
    left_size = len(interface_lists.get(candidate["left_chain"], []))
    right_size = len(interface_lists.get(candidate["right_chain"], []))
    left_count = int(left.get("match_count", 0) or 0)
    right_count = int(right.get("match_count", 0) or 0)
    left_coverage = 100.0 * left_count / left_size if left_size else None
    right_coverage = 100.0 * right_count / right_size if right_size else None

    left_hotspots = filter_assets["hotspots_by_chain"].get(candidate["left_chain"], [])
    right_hotspots = filter_assets["hotspots_by_chain"].get(candidate["right_chain"], [])
    left_hotspot = evaluate_hotspots(left.get("match_dict", {}), left_hotspots)
    right_hotspot = evaluate_hotspots(right.get("match_dict", {}), right_hotspots)

    contacts = transformation._protocol_contacts(
        filter_assets, candidate["left_chain"], candidate["right_chain"]
    )
    contact_count = count_matched_complementary_contacts(
        left.get("match_dict", {}), right.get("match_dict", {}), contacts
    )
    materialized = materialize_candidate(candidate)
    clash_count = materialized["clash_count"]

    available = bool(left.get("_available") and right.get("_available"))
    add_gate(cid, "alignment_record_available", PASS if available else FAIL,
             f"left={left.get('_available')}, right={right.get('_available')}", "both records available")

    fields_ok = transform_fields_valid(left) and transform_fields_valid(right)
    add_gate(cid, "required_transform_fields", PASS if fields_ok else FAIL,
             f"left={transform_fields_valid(left)}, right={transform_fields_valid(right)}", "3-vector + 3x3 matrix on both sides")

    count_ok = left_count >= transformation.MINIMUM_RESIDUE_MATCH_COUNT and right_count >= transformation.MINIMUM_RESIDUE_MATCH_COUNT
    add_gate(cid, "minimum_matched_residues", PASS if count_ok else FAIL,
             f"left={left_count}, right={right_count}", f"both >= {transformation.MINIMUM_RESIDUE_MATCH_COUNT}")

    coverage_ok = (
        left_coverage is not None and right_coverage is not None
        and left_coverage > coverage_requirement(left_size)
        and right_coverage > coverage_requirement(right_size)
    )
    add_gate(cid, "interface_coverage", PASS if coverage_ok else FAIL,
             f"left={left_coverage:.1f}% ({left_count}/{left_size}), right={right_coverage:.1f}% ({right_count}/{right_size})",
             f"strictly > left {coverage_requirement(left_size):.1f}% and right {coverage_requirement(right_size):.1f}%")

    score_ok = score_passes(left) and score_passes(right)
    add_gate(cid, "alignment_score_contract", PASS if score_ok else FAIL,
             f"left TM={float(left.get('tm_score', 0) or 0):.5f}, right TM={float(right.get('tm_score', 0) or 0):.5f}",
             f"both TMalign TM >= {transformation.TM_SCORE_THRESHOLD}")

    hotspot_ok = left_hotspot.passed and right_hotspot.passed
    protocol_active = FILTER_MODE == "published_protocol"
    add_gate(cid, "hotspot_mapping", PASS if hotspot_ok else FAIL,
             f"left={left_hotspot.hotspot_matches}, right={right_hotspot.hotspot_matches}",
             f">= {transformation.MINIMUM_HOTSPOT_MATCH_NUMBER} mapped hotspot on each side", protocol_active)

    contacts_ok = contact_count >= transformation.CONTACT_COUNT_THRESHOLD
    add_gate(cid, "complementary_interface_contacts", PASS if contacts_ok else FAIL,
             contact_count, f">= {transformation.CONTACT_COUNT_THRESHOLD} contacts", protocol_active)

    add_gate(cid, "transformation_materialized", materialized["status"],
             materialized["reason"], "both temporary transformed PDBs written")

    clash_status = NOT_EVALUABLE if clash_count is None else (PASS if clash_count < transformation.MAX_CLASHING_COUNT else FAIL)
    add_gate(cid, "ca_clash_threshold", clash_status, clash_count,
             f"C-alpha pairs < {transformation.CLASHING_DISTANCE} A must total < {transformation.MAX_CLASHING_COUNT}")

    candidate_metrics[cid] = {
        "left_hotspot_pass": left_hotspot.passed,
        "right_hotspot_pass": right_hotspot.passed,
        "contacts_pass": contacts_ok,
        "minimum_count_pass": count_ok,
        "coverage_pass": coverage_ok,
        "score_pass": score_ok,
        "materialized_pass": materialized["status"] == PASS,
        "clash_pass": clash_status == PASS,
    }

gate_ledger = pd.DataFrame(gate_rows)
gate_matrix = gate_ledger.pivot(index="candidate_id", columns="gate", values="status")
display(gate_matrix)
display(gate_ledger)

gate,alignment_record_available,alignment_score_contract,ca_clash_threshold,complementary_interface_contacts,hotspot_mapping,interface_coverage,minimum_matched_residues,required_transform_fields,transformation_materialized
candidate_id,,,,,,,,,
1kcaCH:1FGNH:1TFHA:o1,pass,fail,pass,pass,pass,pass,pass,pass,pass
1kcaCH:1FGNH:1TFHA:o2,pass,fail,fail,pass,pass,pass,pass,pass,pass


,candidate_id,gate,status,observed,criterion,active_in_selected_mode
0,1kcaCH:1FGNH:1TFHA:o1,alignment_record_available,pass,"left=True, right=True",both records available,True
1,1kcaCH:1FGNH:1TFHA:o1,required_transform_fields,pass,"left=True, right=True",3-vector + 3x3 matrix on both sides,True
2,1kcaCH:1FGNH:1TFHA:o1,minimum_matched_residues,pass,"left=17, right=19",both >= 15,True
3,1kcaCH:1FGNH:1TFHA:o1,interface_coverage,pass,"left=54.8% (17/31), right=86.4% (19/22)",strictly > left 50.0% and right 50.0%,True
4,1kcaCH:1FGNH:1TFHA:o1,alignment_score_contract,fail,"left TM=0.24791, right TM=0.30426",both TMalign TM >= 0.5,True
5,1kcaCH:1FGNH:1TFHA:o1,hotspot_mapping,pass,"left=6, right=6",>= 1 mapped hotspot on each side,True
6,1kcaCH:1FGNH:1TFHA:o1,complementary_interface_contacts,pass,9,>= 5 contacts,True
7,1kcaCH:1FGNH:1TFHA:o1,transformation_materialized,pass,temporary_outputs_written,both temporary transformed PDBs written,True
8,1kcaCH:1FGNH:1TFHA:o1,ca_clash_threshold,pass,0,C-alpha pairs < 3.0 A must total < 5,True
9,1kcaCH:1FGNH:1TFHA:o2,alignment_record_available,pass,"left=True, right=True",both records available,True


### 6. Apply gates cumulatively in the diagnostic order

This order is for interpretation, not a proposed production reorder. It progresses from data sufficiency, through alignment plausibility and protocol support, to geometry execution. Independent outcomes above remain the primary evidence because cumulative counts are order-dependent.

In [7]:
diagnostic_order = [
    "alignment_record_available",
    "required_transform_fields",
    "minimum_matched_residues",
    "interface_coverage",
    "alignment_score_contract",
    "hotspot_mapping",
    "complementary_interface_contacts",
    "transformation_materialized",
    "ca_clash_threshold",
]

status_lookup = gate_ledger.set_index(["candidate_id", "gate"])["status"].to_dict()
active_lookup = gate_ledger.set_index(["candidate_id", "gate"])["active_in_selected_mode"].to_dict()
survivors = set(gate_ledger["candidate_id"].unique())
ladder_rows = [{"step": 0, "gate": "initial cohort", "remaining": len(survivors), "candidate_ids": sorted(survivors)}]

for step, gate in enumerate(diagnostic_order, start=1):
    survivors = {
        cid for cid in survivors
        if (not active_lookup[(cid, gate)]) or status_lookup[(cid, gate)] == PASS
    }
    ladder_rows.append({"step": step, "gate": gate, "remaining": len(survivors), "candidate_ids": sorted(survivors)})

diagnostic_ladder = pd.DataFrame(ladder_rows)
display(diagnostic_ladder)

,step,gate,remaining,candidate_ids
0,0,initial cohort,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
1,1,alignment_record_available,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
2,2,required_transform_fields,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
3,3,minimum_matched_residues,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
4,4,interface_coverage,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
5,5,alignment_score_contract,0,[]
6,6,hotspot_mapping,0,[]
7,7,complementary_interface_contacts,0,[]
8,8,transformation_materialized,0,[]
9,9,ca_clash_threshold,0,[]


### 7. Replay the current production order

`process_pair_for_template()` currently evaluates the published-protocol pair gate first, then the combined per-side alignment threshold helper, then writes transformed structures, then rejects clashes. In `geometry_only_experimental`, the protocol pair gate and hotspot checks are bypassed.

In [8]:
production_rows = []
for candidate in candidates:
    cid = candidate["candidate_id"]
    metrics = candidate_metrics[cid]
    checks = []
    if FILTER_MODE == "published_protocol":
        checks.append(("evaluate_protocol_candidate", metrics["left_hotspot_pass"] and metrics["right_hotspot_pass"] and metrics["contacts_pass"]))
    else:
        checks.append(("evaluate_protocol_candidate (bypassed)", True))

    # alignment_passes_thresholds repeats hotspot checks in published mode.
    alignment_ok = (
        metrics["minimum_count_pass"]
        and metrics["coverage_pass"]
        and metrics["score_pass"]
        and (FILTER_MODE != "published_protocol" or (metrics["left_hotspot_pass"] and metrics["right_hotspot_pass"]))
    )
    checks.extend([
        ("alignment_passes_thresholds", alignment_ok),
        ("create_transformed_pair", metrics["materialized_pass"]),
        ("pair_has_acceptable_clashes", metrics["clash_pass"]),
    ])

    first_failure = next((name for name, passed in checks if not passed), None)
    production_rows.append(
        {
            "candidate_id": cid,
            "selected_mode": FILTER_MODE,
            "production_result": "generated" if first_failure is None else "rejected",
            "first_failed_production_check": first_failure,
            "check_trace": " -> ".join(f"{name}:{'pass' if passed else 'fail'}" for name, passed in checks),
        }
    )

production_replay = pd.DataFrame(production_rows)
display(production_replay)

,candidate_id,selected_mode,production_result,first_failed_production_check,check_trace
0,1kcaCH:1FGNH:1TFHA:o1,published_protocol,rejected,alignment_passes_thresholds,evaluate_protocol_candidate:pass -> alignment_...
1,1kcaCH:1FGNH:1TFHA:o2,published_protocol,rejected,alignment_passes_thresholds,evaluate_protocol_candidate:pass -> alignment_...


### 8. Measure each gate's unique rescue potential

In [9]:
active_gates = [
    gate for gate in diagnostic_order
    if any(active_lookup[(cid, gate)] for cid in gate_matrix.index)
]

def passes_gate_set(candidate_id: str, gates: list[str]) -> bool:
    return all(status_lookup[(candidate_id, gate)] == PASS for gate in gates)

full_passers = {cid for cid in gate_matrix.index if passes_gate_set(cid, active_gates)}
rescue_rows = []
for omitted_gate in active_gates:
    remaining_gates = [gate for gate in active_gates if gate != omitted_gate]
    passers_without_gate = {cid for cid in gate_matrix.index if passes_gate_set(cid, remaining_gates)}
    rescued = sorted(passers_without_gate - full_passers)
    rescue_rows.append(
        {
            "omitted_gate": omitted_gate,
            "full_pipeline_passers": len(full_passers),
            "passers_without_gate": len(passers_without_gate),
            "rescued_count": len(rescued),
            "rescued_candidate_ids": rescued,
        }
    )

leave_one_out = pd.DataFrame(rescue_rows).sort_values(
    ["rescued_count", "omitted_gate"], ascending=[False, True]
)
display(leave_one_out)

,omitted_gate,full_pipeline_passers,passers_without_gate,rescued_count,rescued_candidate_ids
4,alignment_score_contract,0,1,1,[1kcaCH:1FGNH:1TFHA:o1]
0,alignment_record_available,0,0,0,[]
8,ca_clash_threshold,0,0,0,[]
6,complementary_interface_contacts,0,0,0,[]
5,hotspot_mapping,0,0,0,[]
3,interface_coverage,0,0,0,[]
2,minimum_matched_residues,0,0,0,[]
1,required_transform_fields,0,0,0,[]
7,transformation_materialized,0,0,0,[]


### 9. Optional one-variable TM-score sensitivity check

In [10]:
# Descriptive only: all other gates and production defaults remain unchanged.
experimental_tm_thresholds = [0.20, 0.25, 0.30, 0.40, 0.50]
sensitivity_rows = []
for threshold in experimental_tm_thresholds:
    passing = []
    for candidate in candidates:
        left_tm = float(candidate["left_alignment"].get("tm_score", 0) or 0)
        right_tm = float(candidate["right_alignment"].get("tm_score", 0) or 0)
        if left_tm >= threshold and right_tm >= threshold:
            passing.append(candidate["candidate_id"])
    sensitivity_rows.append(
        {"diagnostic_tm_threshold": threshold, "score_gate_passers": len(passing), "candidate_ids": passing}
    )
tm_sensitivity = pd.DataFrame(sensitivity_rows)
display(tm_sensitivity)

,diagnostic_tm_threshold,score_gate_passers,candidate_ids
0,0.20,2,"[1kcaCH:1FGNH:1TFHA:o1, 1kcaCH:1FGNH:1TFHA:o2]"
1,0.25,1,[1kcaCH:1FGNH:1TFHA:o2]
2,0.30,0,[]
3,0.40,0,[]
4,0.50,0,[]


## Checks

The assertions below make silent cohort loss, duplicated gate records, and mismatch between the long and wide ledgers explicit.

In [11]:
expected_candidates = {candidate["candidate_id"] for candidate in candidates}
assert set(gate_ledger["candidate_id"]) == expected_candidates
assert not gate_ledger.duplicated(["candidate_id", "gate"]).any()
assert set(gate_matrix.columns) == set(diagnostic_order)
assert diagnostic_ladder.iloc[0]["remaining"] == len(expected_candidates)
assert diagnostic_ladder["remaining"].is_monotonic_decreasing
assert set(production_replay["candidate_id"]) == expected_candidates
print(f"Checks passed: {len(expected_candidates)} candidates x {len(diagnostic_order)} gates, with no silent drops.")

Checks passed: 2 candidates x 9 gates, with no silent drops.


## Takeaways

In [12]:
independent_failures = (
    gate_ledger[gate_ledger["active_in_selected_mode"]]
    .assign(failed=lambda frame: frame["status"] != PASS)
    .groupby("gate", as_index=False)["failed"]
    .sum()
    .rename(columns={"failed": "failed_candidates"})
    .sort_values(["failed_candidates", "gate"], ascending=[False, True])
)
display(independent_failures)

first_zero = diagnostic_ladder.loc[diagnostic_ladder["remaining"].eq(0), "gate"]
print("Selected diagnostic mode:", FILTER_MODE)
print("Initial candidates:", len(expected_candidates))
print("Candidates passing every active gate:", int(diagnostic_ladder.iloc[-1]["remaining"]))
print("First cumulative gate reducing survivors to zero:", first_zero.iloc[0] if not first_zero.empty else "none")
print("Interpretation boundary: this one retained case localizes attrition; it does not calibrate a new production threshold.")

,gate,failed_candidates
1,alignment_score_contract,2
2,ca_clash_threshold,1
0,alignment_record_available,0
3,complementary_interface_contacts,0
4,hotspot_mapping,0
5,interface_coverage,0
6,minimum_matched_residues,0
7,required_transform_fields,0
8,transformation_materialized,0


Selected diagnostic mode: published_protocol
Initial candidates: 2
Candidates passing every active gate: 0
First cumulative gate reducing survivors to zero: alignment_score_contract
Interpretation boundary: this one retained case localizes attrition; it does not calibrate a new production threshold.


## Next steps

- To inspect geometry-only behavior, change `FILTER_MODE` to `geometry_only_experimental`, restart the kernel, and run all cells.
- To test another frozen case, change `RUN_ROOT`, `TEMPLATE_ID`, and `TEMPLATE_CHAINS` together and verify the provenance table before interpreting results.
- To test a threshold, change exactly one threshold through an environment variable **before importing** `src.transformation`, then restart and run all. Keep the resulting notebook as diagnostic evidence; do not silently promote the override to a production default.
- For calibration claims, repeat across a representative frozen cohort and stratify by aligner. TMalign TM-scores and MultiProt's native match/coverage contract are not interchangeable.